# 02 — Exploratory Data Analysis (HUPA-UCM)

**Reading this notebook without a programming background.**
Every code cell below is followed by a *plain-language* interpretation cell.
You can skip the code: the markdown explains what was computed, what the result is, and why it matters for the forecasting model.

**What this notebook answers (seven questions from `skills/SKILL.md` §2).**

1. *Structural quality* — Are the released files internally consistent? Any missing values, duplicate timestamps, or irregular sampling?
2. *Glucose distribution* — How are glucose readings spread? How much time do patients spend in hypoglycaemia, in range, and in hyperglycaemia?
3. *Glucose dynamics* — How fast does glucose change? How long does the past keep being informative about the future (the lookback question)?
4. *Per-patient heterogeneity* — Are patients similar enough that one population model is enough, or so different that a patient-specific context is required?
5. *Circadian and weekly patterns* — Do glucose levels change with time of day or day of week?
6. *Multimodal evidence* — Do bolus insulin, carbohydrate intake, and physical activity actually move future glucose? Pooled, by subtype, and per patient.
7. *Forecasting feasibility* — Does the dataset contain enough usable input windows for deep learning?

All numbers in this notebook are reproducible by running `src/hupa_eda.py` from a fresh Colab runtime.

## 1. Environment

The cell below detects whether the notebook is running on Google Colab or locally. On Colab it mounts the user's Drive; locally it uses the current working directory as the project root. Every file path below is derived from `BASE_PATH`, so the notebook never hard-codes a Windows or Mac path.

In [ ]:
import os
import sys
from pathlib import Path

try:
    import google.colab  # type: ignore
    IN_COLAB = True
    from google.colab import drive
    drive.mount('/content/drive')
    BASE_PATH = '/content/drive/MyDrive/glucose-thesis/'
except Exception:
    IN_COLAB = False
    BASE_PATH = os.path.abspath(os.path.join(os.getcwd()))

BASE_PATH = Path(BASE_PATH)
SRC_PATH = BASE_PATH / 'src'
if str(SRC_PATH) not in sys.path:
    sys.path.insert(0, str(SRC_PATH))

print('IN_COLAB:', IN_COLAB)
print('BASE_PATH:', BASE_PATH)

## 2. Run the EDA pipeline

All heavy computation lives in `src/hupa_eda.py` so that the same code runs from the notebook *and* from a terminal. The function `run_eda` reads every patient Excel file, computes the analyses described in this notebook, and saves results to `outputs/tables/`, `outputs/figures/`, and `outputs/hupa_eda_summary.txt`. It also returns a dictionary mapping each artefact name to its file path, so the rest of the notebook just loads the precomputed tables.

In [ ]:
import pandas as pd
from IPython.display import Image, display
from hupa_eda import run_eda

outputs = run_eda(BASE_PATH)
for name, path in outputs.items():
    print(f'{name}: {path}')

## 4.1 Structural and data-quality check

**Question.** Before any modelling, we need to know whether the released table is internally consistent: same eight columns for every patient, no missing values inside the recording window, no duplicate timestamps, and a strict 5-minute step between consecutive rows. If any of these fail, the rest of the pipeline cannot be trusted.

**Method.** The function `structural_quality_summary` loops over every patient and computes, for each one: number of rows, time span, number of missing values, number of duplicate timestamps, the share of consecutive timestamps that are exactly 300 seconds apart, basic glucose statistics, the share of records in each glycaemic zone, the share of records that hit the sensor floor (`==40 mg/dL`) or ceiling (`>400`), and a count of insulin/carb/activity events.

**How to read the table below.** Each row is one patient. `n_rows` is the total number of 5-minute samples for that patient. `strict_5min_pct` should be exactly 100 (every consecutive pair of timestamps is 300 seconds apart). `missing_total` should be 0. The three columns ending in `_pct` are the percentage of time the patient spent in hypoglycaemia, in range, and in hyperglycaemia.

In [ ]:
overall = pd.read_csv(outputs['overall'])
patient_summary = pd.read_csv(outputs['patient_summary'])

display(overall.T)
patient_summary.sort_values('n_rows', ascending=False).head(10)[[
    'participant_id', 'n_rows', 'duration_days',
    'missing_total', 'duplicate_timestamps', 'strict_5min_pct',
    'glucose_mean', 'hypo_pct', 'tir_pct', 'hyper_pct',
    'basal_positive_pct', 'bolus_event_count', 'carb_event_count'
]]

**Plain-language summary.**
All 25 patients pass the structural check: no missing values, no duplicate timestamps, and `strict_5min_pct` is 100 for everyone. The dataset is already aligned to a clean 5-minute grid — this is the work the dataset authors did using their `glUCModel` tool. We do **not** need to spend methodological effort on gap filling or alignment; we inherit those choices.

The main structural concern is **not missingness, but imbalance across patients**. HUPA0027P alone contributes 165,306 of the 309,392 records (53%), and the top three patients combined contribute 75%. If we did a naïve random split of the pooled records into training and test, we would essentially be evaluating the model on HUPA0027P versus HUPA0027P. The split therefore has to be done at the patient level, not at the row level.

## 4.2 Glucose distribution and glycaemic zones

**Question.** What do typical glucose values look like, and how much of the time is spent in each clinically meaningful zone? This determines both the *output range* the model has to predict and where its errors will matter most for safety.

**Method.** We pool all 309,392 glucose readings into a single histogram, then for each patient we count the share of time in three standard zones based on millimetre-of-glucose-per-decilitre (mg/dL) thresholds from Battelino et al. 2019:

- *Hypoglycaemia* — glucose `< 70 mg/dL`. Dangerous: can cause confusion, seizures, or loss of consciousness.
- *In range* (TIR, *time in range*) — `70 ≤ glucose ≤ 180`. The clinical goal.
- *Hyperglycaemia* — glucose `> 180 mg/dL`. Chronic damage to eyes, kidneys, nerves over time.

**Two ways to summarise the cohort.** *Row-weighted* statistics pool all rows together, so HUPA0027P (53% of rows) drives the answer. *Patient-averaged* statistics treat every patient as one observation and average the per-patient percentages, so HUPA0027P weighs the same as every other patient. The two conventions can disagree by ~10 percentage points on this cohort (see §3.7 of `report.md`). We always report both.

In [ ]:
print(f"Row-weighted   : hypo {overall['hypo_pct'].iloc[0]:.2f}%  TIR {overall['tir_pct'].iloc[0]:.2f}%  hyper {overall['hyper_pct'].iloc[0]:.2f}%")
print(f"Patient-averaged: hypo {overall['patient_avg_hypo_pct'].iloc[0]:.2f}%  TIR {overall['patient_avg_tir_pct'].iloc[0]:.2f}%  hyper {overall['patient_avg_hyper_pct'].iloc[0]:.2f}%")
print(f"Sensor floor (glucose==40 mg/dL): {overall['low_cap_pct'].iloc[0]:.2f}% of records")
print(f"Sensor ceiling (glucose>400 mg/dL): {overall['high_extreme_pct'].iloc[0]:.2f}% of records")
display(Image(filename=str(outputs['figure_glucose_distribution'])))

**Plain-language summary.**
Most readings sit between roughly 80 and 200 mg/dL with a long upper tail toward 400. The two dashed lines on the left panel mark the clinical zone boundaries at 70 (hypoglycaemia) and 180 (hyperglycaemia). The cohort spends roughly 7% in hypoglycaemia, ~60% in range, and ~32% in hyperglycaemia under the patient-averaged convention.

Two things matter for the model:
1. **Hypoglycaemia is a minority class.** A model that minimises mean error can still be wrong in the most safety-critical region. We will have to report errors *by glycaemic zone*, and consider a loss function that penalises hypoglycaemia errors more.
2. **Sensor caps are not measurements.** The FreeStyle Libre 2 reports `LO` for glucose under 40 mg/dL and `HI` for over 400; after processing those become exactly `40` and `>400`. We will flag them with `glucose_low_cap` and `glucose_high_extreme` indicator features so the model can treat them differently.

## 4.3 Glucose dynamics and the lookback window

**Question.** How much *past* glucose does the model need to see to predict the next 30/60/90 minutes? Too little and we lose context; too much and we add noise plus training cost.

**Method.** Two standard time-series tools answer this.

- **Velocity.** The 5-minute change in glucose (`glucose(t) − glucose(t−5min)`). Tells us how fast glucose typically swings. We report the 50th/90th/95th/99th percentiles of the *absolute* velocity — the typical rate of change and the rate of change in rare but rapid moments.
- **Autocorrelation (ACF) and partial autocorrelation (PACF).** ACF at lag *k* is how strongly glucose now is linearly related to glucose *k* minutes ago. PACF removes the indirect influence of intermediate lags and keeps only the direct relationship. A high ACF at 120 minutes means "glucose two hours ago still says something about glucose now"; a near-zero PACF at lag 5 minutes would mean "once you know the most recent value, earlier values add no new direct information". We use ACF to choose the *lookback window* and PACF to argue for short-lag features (e.g. velocity, rolling mean) inside that window.

In [ ]:
velocity = pd.read_csv(outputs['velocity'])
acf = pd.read_csv(outputs['acf'])
pacf = pd.read_csv(outputs['pacf'])
display(velocity.T)
display(
    acf.groupby('lag_minutes')['acf'].mean()
    .loc[[30, 60, 90, 120, 180, 225]].to_frame('mean_acf_across_patients')
)
display(pacf.groupby('lag_minutes')['pacf'].mean().head(8).to_frame('mean_pacf'))
display(Image(filename=str(outputs['figure_acf'])))

**Plain-language summary.**
Typical 5-minute glucose changes are small: the median absolute change is about 1.7 mg/dL per 5 minutes. But the long tail matters — the 99th percentile is 17 mg/dL per 5 minutes, i.e. the *fastest 1% of moments* swing fast enough to cross between zones in well under an hour. This justifies including velocity and acceleration as features.

The autocorrelation curve falls slowly: mean ACF is **0.50 at 120 minutes** and **0.30 at 180 minutes**, and only drops under 0.2 around **225 minutes**. Past glucose remains informative for hours. The PACF, in contrast, drops below ~0.1 already at lag 15 minutes — so only the most recent values carry *direct* information; older values matter only through their cumulative trend.

**Decisions for the model.**
- Primary lookback: **24 steps = 120 minutes** (still substantial ACF, manageable training cost).
- Ablation lookback: **36 steps = 180 minutes** (test whether more history helps).
- Include short-lag derived features (velocity, acceleration, rolling means at 30/60/120 minutes) so that the model does not need to rediscover these inside its first layer.

## 4.4 Per-patient heterogeneity

**Question.** Are the 25 patients similar enough that one shared model is sensible, or so different that the model needs a per-patient context vector (a *static embedding branch*)?

**Method.** Two side-by-side views.
1. *Zone composition per patient* — stacked bars showing what fraction of each patient's records fall in hypoglycaemia, in range, and hyperglycaemia. Patients are sorted by mean glucose so that the left of the chart is the best-controlled, the right is the most hyperglycaemic.
2. *ACF half-life per patient* — for each patient, the first lag at which their personal ACF drops under 0.5. A short half-life (e.g. 75 min) means glucose dynamics are fast and unstable; a long one (e.g. 260 min) means glucose drifts slowly. If patients differ a lot on this metric, the same lookback length will mean different things to different patients.

In [ ]:
halflife = pd.read_csv(outputs['acf_halflife'])
display(halflife.sort_values('first_lag_below_threshold_minutes'))
display(Image(filename=str(outputs['figure_per_patient_heterogeneity'])))

**Plain-language summary.**
Patients are *very* different from each other.
- HUPA0002P spends **24% of the time below 70 mg/dL** (severe hypoglycaemic burden); HUPA0009P spends ~0%.
- HUPA0017P spends almost two-thirds in hyperglycaemia; HUPA0022P barely 6%.
- ACF half-life ranges from **75 minutes (HUPA0019, HUPA0021)** — glucose changes its character fast — to **260 minutes (HUPA0009)** — very smooth, slow drift.

This is direct evidence that the model needs *patient-aware context*. A shared trunk that processes the past glucose plus a small static branch carrying HbA1c, age, treatment (CSII/MDI), BMI, and derived per-patient statistics (mean glucose, hypo fraction, modality availability flags) should fuse better than a single one-size-fits-all model. The patient embedding branch is therefore not a stylistic choice; it is justified by this heterogeneity.

## 4.5 Circadian and weekly patterns

**Question.** Do glucose levels change systematically with the time of day or the day of the week? If yes, the model should be told the *hour* and the *weekday* via cyclical features (sine/cosine of hour, sine/cosine of weekday).

**Method.** Aggregate glucose by hour-of-day and by day-of-week and report mean, median, and per-zone proportions for each bucket.

*Why cyclical encoding.* Hour 23 is one step away from hour 0, not 23 steps. Encoding as `sin(2π·h/24)` and `cos(2π·h/24)` preserves this circular geometry; a raw integer feature would suggest 23 and 0 are far apart.

In [ ]:
circadian = pd.read_csv(outputs['circadian'])
dow = pd.read_csv(outputs['dayofweek'])
display(circadian.head(24))
display(dow[['dayofweek_label', 'glucose_mean', 'tir_pct', 'hypo_pct', 'hyper_pct']])
display(Image(filename=str(outputs['figure_circadian'])))
display(Image(filename=str(outputs['figure_dayofweek_velocity'])))

**Plain-language summary.**
Glucose has a clear time-of-day signature (dawn rise, post-meal peaks) — visible in the circadian profile. Day-of-week effects are smaller but real: Monday and Friday have the highest mean glucose (~145–146), Wednesday the lowest (~138). Sunday shows the lowest TIR (~70%).

Both effects are modest in magnitude, but consistent and cheap to add as features. They will be included in the model with cyclical (`sin`/`cos`) encoding, and we will test their contribution in the ablation experiments (M0 → M1) defined in `CLAUDE.md`.

## 4.6 Multimodal evidence — peri-event glucose response

**Question.** Do bolus insulin, carbohydrate intake, and physical activity actually move future glucose? Should we include them in the model?

**Why we do *not* use a Pearson correlation here.** The naive approach — "correlate the bolus column at time *t* with glucose at time *t+60min*" — fails on this data. Bolus is recorded only at ~1% of 5-minute bins (events are sparse), so the column is mostly zeros. Pearson's *r* between a mostly-zero stream and glucose is structurally near zero, regardless of any physiological effect that exists when an event *does* happen. An earlier lagged-Pearson screen on this cohort returned absolute *r* values below 0.09 for every lag inspected — the algebra of sparse vectors, not evidence about insulin. The screen has been retired; the peri-event analysis below is what we use instead.

**What we do instead — peri-event analysis.** For each event type (any bolus, any carb entry, any 5-minute bin with ≥100 steps), find every time point *t* where the event happened and measure `glucose(t+H) − glucose(t)` for H = 30, 60, 90, 120 minutes. Average across all such events. Then, to filter out background drift, sample the *same number* of *non-event* time points from the *same patient* and compute the same Δglucose. The difference between the event-triggered Δ and the control Δ is the cleanest signal we can compute without a randomised trial: it asks "what happens after this event, *over and above* what would have happened anyway in this patient?"

In [ ]:
peri = pd.read_csv(outputs['peri_event'])
display(peri[[
    'event_type', 'horizon_minutes', 'n_events',
    'mean_delta_event', 'mean_delta_control', 'mean_delta_diff'
]])
display(Image(filename=str(outputs['figure_peri_event'])))

**Plain-language summary.**
All three modalities show a physiologically meaningful effect that is invisible under naïve correlation:

- **Bolus insulin (n=3,586 events).** Glucose first rises slightly in the first 30 min (insulin boluses usually accompany meals), then drops sharply: **−9 mg/dL at 90 min** and **−15 mg/dL at 120 min** above control. This matches the known onset (15 min), peak (~90 min), and duration of rapid-acting insulin.
- **Carbohydrate intake (n=2,641 events).** Glucose rises monotonically to **+15 mg/dL above control by 60 min** and stays elevated. This is the canonical post-meal absorption curve.
- **High-activity bin (≥100 steps in 5 min, n=29,870 events).** Glucose drifts **3–5 mg/dL below control over 30–120 min**. Smaller magnitude than insulin or carbs, but consistent and physiologically expected (exercise improves insulin sensitivity).

**Decision for the model.** All three modalities carry usable signal and will be kept in the candidate feature set. The naive correlation result that some prior glucose-forecasting papers report should not be taken as evidence against multimodal models; it is the wrong statistic for sparse event streams. The actual usefulness of each modality will still need to be verified via the M0→M4 modality-ablation ladder in §5 — peri-event evidence is necessary but not sufficient.

## 4.7 Velocity by glycaemic zone

**Question.** Does glucose change at different speeds inside different zones? If the dynamics in hyperglycaemia are more volatile than in hypoglycaemia, a single pooled RMSE will hide systematic differences in error.

**Method.** Compute the 5-minute glucose change |Δ| for every record, partition by the current zone (hypo / in range / hyper), and report the distribution percentiles. We also rerun the calculation excluding sensor-cap rows (`glucose == 40` or `glucose > 400`) to check whether stuck-at-floor segments artificially deflate the hypoglycaemic quantiles.

In [ ]:
vz = pd.read_csv(outputs['velocity_by_zone'])
vz_filt = pd.read_csv(outputs['velocity_by_zone_filtered'])
print('Unfiltered (includes sensor caps):')
display(vz)
print('Filtered (excludes glucose==40 and >400):')
display(vz_filt)

**Plain-language summary.**
Dynamics differ substantially by zone:
- In **hyperglycaemia**, the 99th-percentile absolute 5-minute change is ~25 mg/dL.
- In **in-range** records it is ~16 mg/dL.
- In **hypoglycaemia** it is only ~11 mg/dL.

Hyperglycaemic windows are roughly **2× more volatile** than hypoglycaemic windows on the tail. A pooled mean-squared-error metric will therefore mostly measure the model's behaviour in hyperglycaemic regions — even though hypoglycaemic mistakes are clinically the worst. This is direct empirical justification for (a) zone-stratified evaluation in §8 and (b) the asymmetric loss with a hypoglycaemia penalty in §5.3.

**Sensor-floor artefact check.** Removing the rows at the sensor floor (`glucose == 40`) and ceiling (`glucose > 400`) drops the hypoglycaemic count from 20,373 to 19,199 records — 5.8% of hypoglycaemic readings sit at the floor. But the quantiles barely move: P50 stays 1.00, P95 stays 5.67, P99 stays 11.00. The 90th percentile shifts only from 4.00 to 4.33 mg/dL per 5 minutes. The sensor cap therefore has a real but bounded effect on the velocity statistics; engineered velocity features can be built on the raw glucose series without special handling beyond the existing censoring indicator flag.

### 4.6.2 Per-patient variance of the response

**Question.** The pooled peri-event statistics above weight by *events*, so HUPA0027P alone contributes more than half. Does the response actually look the same in every patient, or are some patients very different from the cohort average?

**Method.** Compute the mean Δglucose at each horizon **per patient first**, then aggregate across patients (mean ± SD across patients). Saved to `hupa_eda_peri_event_per_patient.csv`.

In [ ]:
peri_pp = pd.read_csv(outputs['peri_event_per_patient'])
display(peri_pp[[
    'event_type', 'horizon_minutes', 'n_patients',
    'patient_mean_diff', 'patient_std_diff',
    'patient_min_diff', 'patient_max_diff'
]])

**Plain-language summary — per-patient view.**

- **Bolus insulin** at +120 min: cross-patient mean **−9.7 mg/dL** with SD **20.8** and range from −43 to +26 across the 22 patients who recorded any bolus. The SD is *more than twice* the mean.
- **Carbohydrate intake** at +120 min: cross-patient mean **+15.2 mg/dL** with SD **22.9** and range from −29 to +68 across 22 patients.
- **High activity (≥100 steps)** at +120 min: cross-patient mean **−8.0 mg/dL**, SD 10.6, range −32 to +7 — every one of the 25 patients had high-activity bins, so this row is the most representative of cohort-wide behaviour.

**Two implications.**
1. The response to bolus and carbohydrate **varies wildly across patients** — the cohort-average effect is *not* a good predictor of any individual patient's response. The model architecture has to allow the modality branches to be modulated by patient context (cross-attention or gated fusion with the static-embedding branch), not added as a fixed offset.
2. Three patients (HUPA0011P, HUPA0015P, HUPA0018P) record **no bolus events at all**, and several record no carb events. The model must run with those feature branches off when serving these patients, which is exactly what the `bolus_available` / `carb_available` flags in §3.6.2-3.6.3 are for.

## 4.9 Summary of decisions handed to Steps 3–5

From the EDA above, the following design constraints are now fixed:

1. **Splitting.** Patient-level, chronological. No random splitting of pooled rows. Long-duration patients (HUPA0026/27/28) require special handling because they together hold 75% of records.
2. **Lookback.** 120 minutes primary, 180 minutes ablation. Both supported by the ACF curve.
3. **Features to include.** Past glucose, velocity, acceleration, rolling means, time-of-day (cyclical), day-of-week (cyclical), patient static metadata (HbA1c, age, gender, BMI, treatment), per-patient derived stats. All three modalities (insulin, carb, activity) survive the peri-event test and enter the candidate set.
4. **Bolus and carbohydrate stay as separate features.** The §4.6.1 subtype analysis showed that correction bolus alone produces −30 mg/dL at 120 min and solo carbohydrate alone produces +26 mg/dL at 120 min. Fusing them into a single "meal event" would average away signals of opposite sign and large magnitude.
5. **Flags, not zero-fills.** Censored glucose (==40, >400) and missing modalities (5 patients) become explicit indicator features so the model can route around them.
6. **Evaluation.** Report per horizon (30/60/90 min), per patient, and per glycaemic zone. Pooled RMSE alone is misleading because hyperglycaemic windows dominate the variance.
7. **Why a patient embedding and patient-conditioned modality fusion.** Per-patient ACF half-life ranges from 75 to 260 min, zone composition varies from 0% hypo to 24% hypo, and the cross-patient SD of the peri-event response is roughly twice the mean magnitude (Table 4.6.1). A shared trunk plus a static-embedding branch is justified, and modality branches must be modulated by patient context (cross-attention / gated fusion), not added as a fixed offset.

All artefacts in `outputs/tables/` and `outputs/figures/02_eda_*.png` are regenerated from a fresh runtime by re-running this notebook end-to-end.

In [ ]:
vz = pd.read_csv(outputs['velocity_by_zone'])
display(vz)

**Plain-language summary.**
Dynamics differ substantially by zone:
- In **hyperglycaemia**, the 99th-percentile absolute 5-minute change is ~25 mg/dL.
- In **in-range** records it is ~16 mg/dL.
- In **hypoglycaemia** it is only ~11 mg/dL.

Hyperglycaemic windows are roughly **2× more volatile** than hypoglycaemic windows on the tail. A pooled mean-squared-error metric will therefore mostly measure the model's behaviour in hyperglycaemic regions — even though hypoglycaemic mistakes are clinically the worst. This is direct empirical justification for (a) zone-stratified evaluation in §8 and (b) the asymmetric loss with a hypoglycaemia penalty in §5.3.

## 4.8 Forecasting feasibility

**Question.** Do we have enough usable sliding-window samples to train a deep learning model without running out of data?

**Method.** For each patient and each candidate lookback (120 min = 24 steps, 180 min = 36 steps), count the number of valid sequences. A sequence is valid if both the past window and the longest forecast horizon (90 min = 18 steps ahead) fit inside the patient's recording. Stride-1 means one sequence per timestamp; stride-6 / stride-12 are sub-sampled versions for faster experimentation.

In [ ]:
seq = pd.read_csv(outputs['sequence'])
display(seq.groupby('lookback')[[
    'usable_sequences_stride1', 'usable_sequences_stride6', 'usable_sequences_stride12'
]].sum())
display(seq.sort_values('usable_sequences_stride1', ascending=False).head(10))

**Plain-language summary.**
The cohort yields ~308,000 stride-1 sequences at the 120-minute lookback. Enough for a neural network. But the per-patient breakdown shows the same imbalance flagged in §4.1: **HUPA0027P alone contributes 165k usable sequences** — more than half the total. So while the *total* count is large, the *effective sample size* is smaller, and the training protocol must protect short-duration patients from being overwhelmed (patient-level cross-validation, possibly sample weighting or truncation of HUPA0027P's contribution).

## 4.9 Summary of decisions handed to Steps 3–5

From the EDA above, the following design constraints are now fixed:

1. **Splitting.** Patient-level, chronological. No random splitting of pooled rows. Long-duration patients (HUPA0026/27/28) require special handling because they together hold 75% of records.
2. **Lookback.** 120 minutes primary, 180 minutes ablation. Both supported by the ACF curve.
3. **Features to include.** Past glucose, velocity, acceleration, rolling means, time-of-day (cyclical), day-of-week (cyclical), patient static metadata (HbA1c, age, gender, BMI, treatment), per-patient derived stats. All three modalities (insulin, carb, activity) survive the peri-event test and enter the candidate set.
4. **Flags, not zero-fills.** Censored glucose (==40, >400) and missing modalities (5 patients) become explicit indicator features so the model can route around them.
5. **Evaluation.** Report per horizon (30/60/90 min), per patient, and per glycaemic zone. Pooled RMSE alone is misleading because hyperglycaemic windows dominate the variance.
6. **Why a patient embedding.** Per-patient ACF half-life ranges from 75 to 260 min, and zone composition varies from 0% hypo to 24% hypo. A shared trunk plus a static-embedding branch is justified by this heterogeneity.

All artefacts in `outputs/tables/` and `outputs/figures/02_eda_*.png` are regenerated from a fresh runtime by re-running this notebook end-to-end.